In [ ]:
# imports

from pathlib import Path
import json
import re
import time

import pandas as pd

# colab data folder
DATA_DIR = Path("/content")

# input paths
CASES_FILE = DATA_DIR / "in_scope_QA.csv"
GUIDELINES_FILE = DATA_DIR / "QA_guidelines.csv"
OUT_OF_SCOPE_CASES_FILE = DATA_DIR / "out_of_scope_QA.csv"


In [ ]:
# load and validate data

# input files
library_cases_df = pd.read_csv(CASES_FILE)
guidelines_df = pd.read_csv(GUIDELINES_FILE)
out_of_scope_cases_df = pd.read_csv(OUT_OF_SCOPE_CASES_FILE)

# expected case schema
required_case_columns = [
    "case_id",
    "question_en",
    "question_nl",
    "context",
    "reference_answer_en",
    "reference_answer_nl",
    "expected_answer_points",
    "should_not_include",
]

# expected guideline schema
required_guideline_columns = [
    "guideline_type",
    "guideline_text",
]


def validate_case_dataframe(df: pd.DataFrame, dataframe_name: str) -> None:
    """
    Check whether a case dataframe contains all required columns.
    """

    # schema check
    missing_columns = [col for col in required_case_columns if col not in df.columns]

    if missing_columns:
        raise ValueError(f"{dataframe_name} is missing required columns: {missing_columns}")


def clean_case_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean text fields in a case dataframe.
    """

    cleaned_df = df.copy()

    # text cleanup
    for col in required_case_columns:
        cleaned_df[col] = cleaned_df[col].fillna("").astype(str).str.strip()

    return cleaned_df


def validate_guidelines_dataframe(df: pd.DataFrame) -> None:
    """
    Check whether the guidelines dataframe contains all required columns.
    """

    missing_columns = [col for col in required_guideline_columns if col not in df.columns]

    if missing_columns:
        raise ValueError(f"Guidelines dataframe is missing required columns: {missing_columns}")


def clean_guidelines_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean text fields in the guidelines dataframe.
    """

    cleaned_df = df.copy()

    # text cleanup
    cleaned_df["guideline_type"] = cleaned_df["guideline_type"].fillna("").astype(str).str.strip()
    cleaned_df["guideline_text"] = cleaned_df["guideline_text"].fillna("").astype(str).str.strip()

    return cleaned_df


# run checks
validate_case_dataframe(library_cases_df, "library_cases_df")
validate_case_dataframe(out_of_scope_cases_df, "out_of_scope_cases_df")
validate_guidelines_dataframe(guidelines_df)

# apply cleanup
library_cases_df = clean_case_dataframe(library_cases_df)
out_of_scope_cases_df = clean_case_dataframe(out_of_scope_cases_df)
guidelines_df = clean_guidelines_dataframe(guidelines_df)

print("All files validated and cleaned.")

In [ ]:
def expand_cases_to_items(
    cases_df: pd.DataFrame,
    dataset_name: str,
) -> pd.DataFrame:
    """
    Expand each bilingual case into separate English and Dutch items.
    """

    benchmark_items = []
    case_records = cases_df.to_dict(orient="records")

    # one row per language
    for case in case_records:
        base_case_id = case["case_id"]
        context = case["context"]
        expected_answer_points = case["expected_answer_points"]
        should_not_include = case["should_not_include"]

        # english item
        if case["question_en"]:
            benchmark_items.append({
                "item_id": f"{base_case_id}_EN",
                "base_case_id": base_case_id,
                "dataset": dataset_name,
                "language": "en",
                "question": case["question_en"],
                "context": context,
                "reference_answer": case["reference_answer_en"],
                "expected_answer_points": expected_answer_points,
                "should_not_include": should_not_include,
            })

        # dutch item
        if case["question_nl"]:
            benchmark_items.append({
                "item_id": f"{base_case_id}_NL",
                "base_case_id": base_case_id,
                "dataset": dataset_name,
                "language": "nl",
                "question": case["question_nl"],
                "context": context,
                "reference_answer": case["reference_answer_nl"],
                "expected_answer_points": expected_answer_points,
                "should_not_include": should_not_include,
            })

    return pd.DataFrame(benchmark_items)


# in-scope items
in_scope_test_df = expand_cases_to_items(
    cases_df=library_cases_df,
    dataset_name="in_scope",
)

# out-of-scope items
out_of_scope_test_df = expand_cases_to_items(
    cases_df=out_of_scope_cases_df,
    dataset_name="out_of_scope",
)

# routing benchmark
test_df = pd.concat(
    [in_scope_test_df, out_of_scope_test_df],
    ignore_index=True,
)

print("Case library shape:", in_scope_test_df.shape)
print("Out-of-scope test shape:", out_of_scope_test_df.shape)
print("Combined routing test shape:", test_df.shape)

display(test_df[["item_id", "dataset", "language", "question"]].head(10))

In [ ]:
# lingua setup

%pip install -q "lingua-language-detector"

from lingua import Language, LanguageDetectorBuilder

# current benchmark languages
lingua_detector = (
    LanguageDetectorBuilder
    .from_languages(Language.ENGLISH, Language.DUTCH)
    .build()
)

In [ ]:
# model setup

%pip install -q -U "transformers>=4.38.0" "accelerate" "bitsandbytes>=0.46.1" "sentencepiece"

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

from google.colab import userdata
from huggingface_hub import login

# colab secret
HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Successfully logged in to Hugging Face Hub.")
else:
    print("HF_TOKEN not found in Colab secrets. Add it if the model requires authentication.")

# routing model
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

# gpu if available
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", device)

# 4-bit loading
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

# inference mode
model.eval()

print("Model loaded:", MODEL_ID)

In [ ]:
# generation helper

def generate_text_with_model(
    prompt: str,
    max_new_tokens: int = 80,
) -> str:
    """
    Generate text from a completed prompt.
    Used only for prompt-based routing classifiers.
    """

    # chat-style input
    messages = [
        {
            "role": "user",
            "content": prompt,
        }
    ]

    # qwen chat template
    try:
        formatted_prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    except Exception:
        formatted_prompt = prompt

    # tokenized input
    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096,
    ).to(device)

    input_length = inputs["input_ids"].shape[-1]

    # no gradients
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    # new tokens only
    generated_ids = output_ids[0][input_length:]
    answer = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    return answer


def parse_json_from_text(raw_output: str) -> dict:
    """
    Try to parse a JSON object from model output.
    If the model returns extra text, extract the first JSON-like block.
    """

    # direct json first
    try:
        return json.loads(raw_output)
    except Exception:
        pass

    # extract json block
    try:
        start_index = raw_output.find("{")
        end_index = raw_output.rfind("}") + 1

        if start_index != -1 and end_index != -1:
            json_text = raw_output[start_index:end_index]
            return json.loads(json_text)
    except Exception:
        pass

    # safe empty fallback
    return {}

In [ ]:
# language detection methods

def detect_language_lingua(user_question: str) -> str:
    """
    Detect whether the user question is English or Dutch using Lingua.

    Returns:
    - 'en'
    - 'nl'
    """

    # fast detector
    detected_language = lingua_detector.detect_language_of(user_question)

    if detected_language == Language.DUTCH:
        return "nl"

    if detected_language == Language.ENGLISH:
        return "en"

    # safe fallback
    return "en"


def detect_language_prompt_based(user_question: str) -> str:
    """
    Detect whether the user question is English or Dutch using the loaded model.

    Returns:
    - 'en'
    - 'nl'
    """

    # classifier prompt
    language_prompt = f"""
You are a language detection classifier for a bilingual household energy chatbot.

Classify the user question as exactly one of these language labels:

1. en
The question is written mainly in English.

2. nl
The question is written mainly in Dutch.

User question:
{user_question}

Return only valid JSON with this exact structure:
{{
  "language": "en",
  "reason": "Brief reason."
}}
""".strip()

    # model prediction
    raw_output = generate_text_with_model(
        prompt=language_prompt,
        max_new_tokens=80,
    )

    # parse label
    parsed_output = parse_json_from_text(raw_output)

    detected_language = parsed_output.get("language", "").strip().lower()

    if detected_language in ["en", "nl"]:
        return detected_language

    # lingua fallback
    return detect_language_lingua(user_question)


def detect_language(
    user_question: str,
    method: str = "lingua",
) -> str:
    """
    Unified language detector.

    Supported methods:
    - 'lingua'
    - 'prompt_based'
    """

    # method switch
    if method == "lingua":
        return detect_language_lingua(user_question)

    if method == "prompt_based":
        return detect_language_prompt_based(user_question)

    raise ValueError("Supported language detection methods are: 'lingua' and 'prompt_based'.")

In [ ]:
# scope detection methods

def contains_keyword(text: str, keywords: list[str]) -> bool:
    """
    Check whether text contains a keyword as a full word or phrase.
    This prevents accidental matches such as 'graph' inside 'paragraph'.
    """

    # case-insensitive match
    text_lower = text.lower()

    # avoid partial words
    for keyword in keywords:
        keyword_lower = keyword.lower()

        if " " in keyword_lower or "-" in keyword_lower:
            if keyword_lower in text_lower:
                return True
        else:
            pattern = r"\b" + re.escape(keyword_lower) + r"\b"
            if re.search(pattern, text_lower):
                return True

    return False


def detect_scope_simple(user_question: str) -> str:
    """
    Rule-based scope detector.

    Returns:
    - 'in_scope'
    - 'out_of_scope'
    """

    # normalized question
    question_lower = user_question.lower()

    # domain terms
    energy_keywords = [
        # English
        "electricity", "energy", "usage", "consumption", "solar", "grid",
        "export", "heat pump", "ventilation", "water use", "water usage",
        "anomaly", "detected", "production", "inverter", "meter", "reading",
        "graph", "energy graph", "usage graph", "consumption graph",
        "peak", "peaks", "device", "appliance", "power", "kwh",

        # Dutch
        "elektriciteit", "elektriciteitsverbruik", "energie", "energieverbruik",
        "zonne", "zonnepanelen", "zonneproductie", "teruglevering",
        "warmtepomp", "ventilatie", "waterverbruik", "afwijking",
        "gedetecteerd", "omvormer", "meter", "grafiek", "energiegrafiek",
        "energiepatroon", "energiepatronen", "piek", "pieken",
        "elektriciteitspiek", "elektriciteitspieken",
        "verbruikspiek", "verbruikspieken",
        "apparaat", "verbruik", "stroom", "kwh", "grafiekje",
    ]

    # exclusion terms
    out_of_scope_keywords = [
        # English
        "recipe", "cook", "stock", "invest", "vote", "election", "trip",
        "travel", "python", "code", "debug", "movie", "translate",
        "wi-fi", "wifi", "router", "joke", "headache", "medical",
        "doctor", "medicine", "spanish",

        # Dutch
        "recept", "koken", "aandeel", "beleggen", "stemmen", "verkiezingen",
        "trip", "reis", "python", "code", "debug", "film", "vertalen",
        "spaans", "wifi", "router", "grap", "hoofdpijn", "medisch",
        "dokter", "medicijn",
    ]

    # keyword flags
    has_energy_keyword = contains_keyword(question_lower, energy_keywords)
    has_out_of_scope_keyword = contains_keyword(question_lower, out_of_scope_keywords)

    if has_out_of_scope_keyword and not has_energy_keyword:
        return "out_of_scope"

    if has_energy_keyword and not has_out_of_scope_keyword:
        return "in_scope"

    # conservative conflict
    if has_out_of_scope_keyword and has_energy_keyword:
        return "out_of_scope"

    # unknown fallback
    return "out_of_scope"


def detect_scope_prompt_based(user_question: str) -> str:
    """
    Detect whether a user question is in scope using the loaded model.

    Returns:
    - 'in_scope'
    - 'out_of_scope'
    """

    # classifier prompt
    scope_prompt = f"""
You are a routing classifier for a BeNext household energy monitoring chatbot.

Classify the user question as one of these labels:

1. in_scope
The question is about household energy monitoring, electricity use, energy consumption, solar production, grid export, heat pumps, ventilation, water use, device-related energy behaviour, energy graphs, peaks, anomalies, or follow-up recommendations based on system readings.

2. out_of_scope
The question is about anything else, such as recipes, stocks, medical advice, politics, travel, coding, movies, translation, jokes, Wi-Fi/router support, or general knowledge.

User question:
{user_question}

Return only valid JSON with this exact structure:
{{
  "scope": "in_scope",
  "reason": "Brief reason."
}}
""".strip()

    # model prediction
    raw_output = generate_text_with_model(
        prompt=scope_prompt,
        max_new_tokens=80,
    )

    # parse label
    parsed_output = parse_json_from_text(raw_output)

    scope = parsed_output.get("scope", "").strip()

    if scope in ["in_scope", "out_of_scope"]:
        return scope

    # rule fallback
    return detect_scope_simple(user_question)


def detect_scope(
    user_question: str,
    method: str = "rule_based",
) -> str:
    """
    Unified scope detector.

    Supported methods:
    - 'rule_based'
    - 'prompt_based'
    """

    # method switch
    if method == "rule_based":
        return detect_scope_simple(user_question)

    if method == "prompt_based":
        return detect_scope_prompt_based(user_question)

    raise ValueError("Supported scope detection methods are: 'rule_based' and 'prompt_based'.")

In [ ]:
# routing evaluation helpers

def get_true_scope(dataset_name: str) -> str:
    """
    Convert dataset name into expected scope label.
    """

    # expected label
    if dataset_name == "out_of_scope":
        return "out_of_scope"

    if dataset_name == "in_scope":
        return "in_scope"

    return "in_scope"

def summarize_detection_results(
    results_df: pd.DataFrame,
    method_column: str,
    correct_column: str,
) -> pd.DataFrame:
    """
    Summarize detection results by method.
    """

    # method summary
    summary_df = (
        results_df
        .groupby(method_column)
        .agg(
            n_items=("item_id", "count"),
            accuracy=(correct_column, "mean"),
            avg_latency_seconds=("latency_seconds", "mean"),
            n_errors=("error_message", lambda x: (x != "").sum()),
        )
        .reset_index()
    )

    return summary_df

In [ ]:
# evaluate language detection

def evaluate_language_detection_on_dataframe(
    input_df: pd.DataFrame,
    language_detection_methods: list[str],
) -> pd.DataFrame:
    """
    Evaluate language detection on the full dataframe.

    True label:
    - input_df["language"]

    Predicted label:
    - detect_language(question, method)
    """

    rows = []

    # progress counter
    total_items = len(input_df)

    # method loop
    for method in language_detection_methods:
        print("=" * 80)
        print(f"Evaluating language detection method: {method}")
        print("=" * 80)

        # item loop
        for row_number, (_, row) in enumerate(input_df.iterrows(), start=1):
            item_id = row.get("item_id", f"row_{row_number}")
            question = row.get("question", "")
            true_language = row.get("language", "")

            print(f"Language detection {row_number}/{total_items}: {item_id} ({method})")

            # item timing
            start_time = time.perf_counter()

            try:
                predicted_language = detect_language(
                    user_question=question,
                    method=method,
                )
                error_message = ""
            except Exception as error:
                # keep failure row
                predicted_language = ""
                error_message = str(error)

            end_time = time.perf_counter()

            # store result
            rows.append({
                "item_id": item_id,
                "base_case_id": row.get("base_case_id", ""),
                "dataset": row.get("dataset", ""),
                "category": row.get("category", ""),
                "question": question,

                "true_language": true_language,
                "predicted_language": predicted_language,
                "language_detection_method": method,
                "language_detection_correct": predicted_language == true_language,

                "latency_seconds": end_time - start_time,
                "error_message": error_message,
            })

    results_df = pd.DataFrame(rows)

    print("\nLanguage detection evaluation finished.")
    print("Results shape:", results_df.shape)

    return results_df


language_detection_results = evaluate_language_detection_on_dataframe(
    input_df=test_df,
    language_detection_methods=["lingua", "prompt_based"],
)

language_detection_summary = summarize_detection_results(
    results_df=language_detection_results,
    method_column="language_detection_method",
    correct_column="language_detection_correct",
)

display(language_detection_summary)

# wrong cases only
display(
    language_detection_results[
        language_detection_results["language_detection_correct"] == False
    ][
        [
            "item_id",
            "dataset",
            "category",
            "question",
            "true_language",
            "predicted_language",
            "language_detection_method",
            "latency_seconds",
            "error_message",
        ]
    ]
)

In [ ]:
# evaluate scope detection

def evaluate_scope_detection_on_dataframe(
    input_df: pd.DataFrame,
    scope_detection_methods: list[str],
) -> pd.DataFrame:
    """
    Evaluate scope detection on the full dataframe.

    True label:
    - baseline -> in_scope
    - out_of_scope -> out_of_scope

    Predicted label:
    - detect_scope(question, method)
    """

    rows = []

    # progress counter
    total_items = len(input_df)

    # method loop
    for method in scope_detection_methods:
        print("=" * 80)
        print(f"Evaluating scope detection method: {method}")
        print("=" * 80)

        # item loop
        for row_number, (_, row) in enumerate(input_df.iterrows(), start=1):
            item_id = row.get("item_id", f"row_{row_number}")
            question = row.get("question", "")
            dataset_name = row.get("dataset", "")
            # benchmark label
            true_scope = get_true_scope(dataset_name)

            print(f"Scope detection {row_number}/{total_items}: {item_id} ({method})")

            # item timing
            start_time = time.perf_counter()

            try:
                predicted_scope = detect_scope(
                    user_question=question,
                    method=method,
                )
                error_message = ""
            except Exception as error:
                # keep failure row
                predicted_scope = ""
                error_message = str(error)

            end_time = time.perf_counter()

            # store result
            rows.append({
                "item_id": item_id,
                "base_case_id": row.get("base_case_id", ""),
                "dataset": dataset_name,
                "category": row.get("category", ""),
                "question": question,

                "true_scope": true_scope,
                "predicted_scope": predicted_scope,
                "scope_detection_method": method,
                "scope_detection_correct": predicted_scope == true_scope,

                "latency_seconds": end_time - start_time,
                "error_message": error_message,
            })

    results_df = pd.DataFrame(rows)

    print("\nScope detection evaluation finished.")
    print("Results shape:", results_df.shape)

    return results_df


scope_detection_results = evaluate_scope_detection_on_dataframe(
    input_df=test_df,
    scope_detection_methods=["rule_based", "prompt_based"],
)

scope_detection_summary = summarize_detection_results(
    results_df=scope_detection_results,
    method_column="scope_detection_method",
    correct_column="scope_detection_correct",
)

display(scope_detection_summary)

# wrong cases only
display(
    scope_detection_results[
        scope_detection_results["scope_detection_correct"] == False
    ][
        [
            "item_id",
            "dataset",
            "category",
            "question",
            "true_scope",
            "predicted_scope",
            "scope_detection_method",
            "latency_seconds",
            "error_message",
        ]
    ]
)